In [4]:
import pandas as pd

In [5]:
df = pd.read_csv('livros.csv')
df.head()

,Book,Author(s),Original language,First published,Approximate sales in millions,Genre
0,A Tale of Two Cities,Charles Dickens,English,1859,200.0,Historical fiction
1,The Little Prince (Le Petit Prince),Antoine de Saint-Exupéry,French,1943,200.0,Novella
2,Harry Potter and the Philosopher's Stone,J. K. Rowling,English,1997,120.0,Fantasy
3,And Then There Were None,Agatha Christie,English,1939,100.0,Mystery
4,Dream of the Red Chamber (紅樓夢),Cao Xueqin,Chinese,1791,100.0,Family saga


In [6]:
df.shape

(174, 6)

In [7]:
df.columns

Index(['Book', 'Author(s)', 'Original language', 'First published',
       'Approximate sales in millions', 'Genre'],
      dtype='object')

In [8]:
traducao_colunas = {
    'Book':'livro',
    'Author(s)':'autor',
    'Original language':'idioma_original',
    'First published':'ano_publicacao',
    'Approximate sales in millions':'vendas',
    'Genre':'genero'
}

In [9]:
df.rename(columns=traducao_colunas, inplace=True)

In [10]:
df.columns

Index(['livro', 'autor', 'idioma_original', 'ano_publicacao', 'vendas',
       'genero'],
      dtype='object')

In [11]:
df.isnull().sum()

,0
livro,0
autor,0
idioma_original,0
ano_publicacao,0
vendas,0
genero,56


In [12]:
autores_unicos = pd.DataFrame(df['autor'].unique(), columns=['autor'])
autores_unicos.head()

,autor
0,Charles Dickens
1,Antoine de Saint-Exupéry
2,J. K. Rowling
3,Agatha Christie
4,Cao Xueqin


In [13]:
autores = pd.DataFrame(df['autor'].unique(), columns=['nome'])
autores.head()

,nome
0,Charles Dickens
1,Antoine de Saint-Exupéry
2,J. K. Rowling
3,Agatha Christie
4,Cao Xueqin


In [14]:
with open('autores.sql', 'w', encoding='utf-8') as arquivo:
    for _, row in autores.iterrows():
        nome = row['nome'].replace("'", "''")
        arquivo.write(f"INSERT INTO autores (nome) VALUES ('{nome}');\n")

In [15]:
df['genero'] = df['genero'].fillna('Unknown')
df.tail(10)

,livro,autor,idioma_original,ano_publicacao,vendas,genero
164,The Joy of Sex,Alex Comfort,English,1972,10.0,Unknown
165,The Gospel According to Peanuts,Robert L. Short,English,1965,10.0,Unknown
166,The Subtle Art of Not Giving a Fuck,Mark Manson,English,2016,10.0,Unknown
167,Life of Pi,Yann Martel,English,2001,10.0,Unknown
168,The Front Runner,Patricia Nell Warren,English,1974,10.0,Unknown
169,The Goal,Eliyahu M. Goldratt,English,1984,10.0,Unknown
170,Fahrenheit 451,Ray Bradbury,English,1953,10.0,Unknown
171,Angela's Ashes,Frank McCourt,English,1996,10.0,Unknown
172,The Story of My Experiments with Truth (સત્યના...,Mohandas Karamchand Gandhi,Gujarati,1929,10.0,Unknown
173,Bridget Jones's Diary,Helen Fielding,English,1996,10.0,Unknown


In [16]:
generos = pd.DataFrame(df['genero'].unique(), columns=['genero'])
generos.head()

,genero
0,Historical fiction
1,Novella
2,Fantasy
3,Mystery
4,Family saga


In [17]:
!pip install -q deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.8 MB/s eta 0:00:00


In [18]:
from deep_translator import GoogleTranslator

generos['genro_pt'] = generos['genero'].apply(lambda x: GoogleTranslator(source='auto', target='pt').translate(x))
generos.head()

,genero,genro_pt
0,Historical fiction,Ficção histórica
1,Novella,Novella
2,Fantasy,Fantasia
3,Mystery,Mistério
4,Family saga,Saga da família


In [19]:
generos['genro_pt'] = generos['genro_pt'].replace('Novella', 'Novela')
generos.head()

,genero,genro_pt
0,Historical fiction,Ficção histórica
1,Novella,Novela
2,Fantasy,Fantasia
3,Mystery,Mistério
4,Family saga,Saga da família


In [20]:
generos['genro_pt'] = generos['genro_pt'].str.lower()
generos.head()

,genero,genro_pt
0,Historical fiction,ficção histórica
1,Novella,novela
2,Fantasy,fantasia
3,Mystery,mistério
4,Family saga,saga da família


In [21]:
with open('generos.sql', 'w', encoding='utf-8') as arquivo:
    for nome in generos['genro_pt']:
        nome_validado = nome.replace("'", "''")
        arquivo.write(f"INSERT INTO generos (nome) VALUES ('{nome_validado}');\n")

In [26]:
autores_unicos['autor_id'] = autores_unicos.index + 1
df = df.merge(autores_unicos, on='autor', how='left')
generos_unicos = pd.DataFrame(df['genero'].unique(), columns=['genero'])
generos_unicos['genero_id'] = generos_unicos.index + 1
df = df.merge(generos_unicos, on='genero', how='left')
df.head()

,livro,autor,idioma_original,ano_publicacao,vendas,genero,genro_pt,autor_id,genero_id
0,A Tale of Two Cities,Charles Dickens,English,1859,200.0,Historical fiction,ficção histórica,1,1
1,The Little Prince (Le Petit Prince),Antoine de Saint-Exupéry,French,1943,200.0,Novella,novela,2,2
2,Harry Potter and the Philosopher's Stone,J. K. Rowling,English,1997,120.0,Fantasy,fantasia,3,3
3,And Then There Were None,Agatha Christie,English,1939,100.0,Mystery,mistério,4,4
4,Dream of the Red Chamber (紅樓夢),Cao Xueqin,Chinese,1791,100.0,Family saga,saga da família,5,5


In [28]:
with open('livros.sql', 'w', encoding='utf-8') as arquivo:
    for _, row in df.iterrows():
        livro = row['livro'].replace("'", "''")
        idioma = row['idioma_original'].replace("'", "''")
        ano_publicacao = row['ano_publicacao']
        vendas = row['vendas']
        autor_id = row['autor_id']
        genero_id = row['genero_id']

        sql = (
            f"INSERT INTO livros (nome, idioma, ano_publicacao, vendas, autor_id, genero_id) "
            f"VALUES ('{livro}', '{idioma}', {ano_publicacao}, {vendas}, {autor_id}, {genero_id});\n"
        )
        arquivo.write(sql)